# MultiModal RAG App for Video Processing With LlamaIndex and LanceDB

## What This Notebook Covers

This notebook builds a multimodal RAG (Retrieval Augmented Generation) system that can watch a YouTube video and answer questions about both the spoken content and the visual content shown on screen.

Most RAG systems work only on text. This notebook goes further by treating video as a first-class data source: it extracts frames as images, transcribes the audio as text, and builds a unified index that stores embeddings for both modalities. When a question arrives, the system retrieves the most relevant text chunks and the most visually relevant frames simultaneously, then passes everything to Gemini 2.5 Flash which can read both text and images to generate a grounded answer.

The notebook is organised into seven categories.

Category 1 installs all required libraries and imports all modules used throughout the notebook.

Category 2 defines all file and folder paths and implements the four helper functions for video downloading, frame extraction, audio extraction, and speech-to-text transcription.

Category 3 executes the full video preprocessing pipeline: download, extract frames, extract audio, transcribe, save text, and clean up temporary files.

Category 4 builds the multimodal vector index by loading all mixed-data files into LlamaIndex, creating separate LanceDB vector stores for text and images, setting up the embedding models, and indexing everything.

Category 5 creates the multimodal retriever, runs queries, inspects retrieved nodes, and visualises the retrieved image frames.

Category 6 assembles the full multimodal prompt, prepares image inputs, and generates the final answer using the Google GenAI SDK with Gemini 2.5 Flash.

Category 7 contains the pipeline architecture diagrams and the conclusion.

## Category 1  Installation and Imports

### Why Each Library Is Needed

llama-index-vector-stores-lancedb provides the LanceDB vector store backend that LlamaIndex uses to persist and search text and image embeddings on disk without needing a separate database server.

llama-index-multi-modal-llms-openai was the original integration for GPT-4 Vision. It is still installed for completeness but is replaced by the Google GenAI SDK in this notebook due to compatibility issues with current LlamaIndex versions.

llama-index-embeddings-clip and the CLIP repository provide the ViT-B/32 vision model that converts image frames into dense vectors. CLIP was trained on image-caption pairs so it produces embeddings where images and their textual descriptions land near each other in vector space, which is exactly what multimodal retrieval requires.

llama-index-readers-file provides SimpleDirectoryReader which scans a folder and loads all supported file types including text files and images into LlamaIndex Document objects.

openai-whisper and faster-whisper provide speech-to-text transcription. faster-whisper is a reimplementation of OpenAI Whisper using CTranslate2 which runs significantly faster on CPU with the int8 compute type.

moviepy is a Python video editing library used here to extract frames as PNG images and to extract the audio track as a WAV file from the downloaded MP4.

yt-dlp is a maintained fork of youtube-dl that downloads YouTube videos reliably. pytube is also imported as a fallback. yt-dlp is preferred because it handles more YouTube formats and is more actively maintained.

pydub, SpeechRecognition, soundfile, and ffmpeg-python are audio processing libraries. ffmpeg-python provides Python bindings for the FFmpeg multimedia framework which moviepy calls internally for audio decoding.

torch and torchvision are required by the CLIP model for neural network inference.

matplotlib and scikit-image are used for displaying retrieved image frames visually.

lancedb is the underlying vector database library. LanceDB stores vectors in the Lance columnar format on local disk, supports hybrid search, and requires no separate server process making it ideal for local development and notebooks.

google-genai is the official Google Generative AI Python SDK. It is used in the final generation step because the LlamaIndex GeminiMultiModal wrapper has compatibility issues with current ImageDocument objects, while the raw SDK call works correctly.

In [ ]:
%pip install llama-index-vector-stores-lancedb
%pip install llama-index-multi-modal-llms-openai
%pip install llama-index-embeddings-clip
%pip install git+https://github.com/openai/CLIP.git
!pip install llama-index-readers-file

In [ ]:
%pip install llama_index
%pip install -U openai-whisper

In [ ]:
%pip install lancedb
%pip install moviepy
%pip install pytube
%pip install pydub
%pip install SpeechRecognition
%pip install ffmpeg-python
%pip install soundfile
%pip install torch torchvision
%pip install matplotlib scikit-image
%pip install ftfy regex tqdm

### What ffmpeg and moviepy Do

ffmpeg is a command-line multimedia framework that can decode virtually any video or audio format. moviepy wraps ffmpeg in a Python API so we can write Pythonic code like clip.audio.write_audiofile() instead of constructing ffmpeg shell commands manually. ffmpeg-python provides low-level Python bindings to ffmpeg for cases where direct control is needed.

moviepy's VideoFileClip class reads the downloaded MP4, exposes the video as a sequence of frames, and exposes the audio track as a separate stream. Both extraction functions in Category 2 rely on this class.

### Importing All Modules

All imports are gathered here so that the notebook fails fast with a clear ImportError if any library is missing, rather than failing mid-pipeline after preprocessing has already run. This also makes it easy to see at a glance which libraries every part of the notebook depends on.

In [ ]:
import sys
print(sys.executable)

In [ ]:
from pathlib import Path
import speech_recognition as sr
from pytube import YouTube
from pprint import pprint
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
import os
print(os.getcwd())

## Category 2  Path Setup and Helper Function Definitions

### Why Paths Are Defined First

Every processing step in this notebook reads from or writes to specific locations on disk. Defining all paths in one place at the top of the pipeline makes them easy to change and prevents inconsistencies where one function writes to a path that another function does not know about.

Path.cwd() returns the current working directory as a pathlib Path object. Using pathlib throughout is preferred over string concatenation because the forward-slash operator builds paths correctly on both Windows and Linux without manual separator handling.

mkdir with parents=True and exist_ok=True creates the folder and any missing parent folders. Passing exist_ok=True prevents an error if the folder already exists from a previous run.

### The video_data Folder

video_data is the staging area for the downloaded MP4 file. It is kept separate from mixed_data because the MP4 is a large intermediate file. After preprocessing is complete the MP4 is no longer needed by LlamaIndex and keeping it separate makes cleanup easier.

### The mixed_data Folder

mixed_data is the output folder that LlamaIndex will later scan with SimpleDirectoryReader. It receives all three outputs of the preprocessing pipeline: PNG image frames, the WAV audio file (temporarily), and the transcribed text file. SimpleDirectoryReader loads every file in this folder so everything in mixed_data automatically becomes part of the multimodal index.

## Define the Video Storage Path

In [ ]:
from pathlib import Path

# Current project directory used as the root for all paths in this notebook
project_dir = Path.cwd()

# video_data: staging folder for the downloaded MP4 file
# Kept separate from mixed_data because the MP4 is not loaded into the LlamaIndex index
output_video_path = project_dir / "video_data"

# Create the folder if it does not already exist
# parents=True: create any missing parent directories
# exist_ok=True: do not raise an error if the folder already exists
output_video_path.mkdir(parents=True, exist_ok=True)

# Full path to the downloaded video file inside video_data
input_video_path = output_video_path / "input_vid.mp4"

## Define the Output Storage Paths

In [ ]:
from pathlib import Path

# mixed_data: the output folder that LlamaIndex will scan with SimpleDirectoryReader
# All three preprocessing outputs go here: PNG frames, the audio WAV, and the text file
# SimpleDirectoryReader loads every file in this folder into the multimodal index
output_folder = project_dir / "mixed_data"

# Create mixed_data if it does not already exist
output_folder.mkdir(parents=True, exist_ok=True)

# Path for the extracted audio file (WAV format required by Whisper)
output_audio_path = output_folder / "output_audio.wav"

# Path for the transcribed text file that stores the speech-to-text output
output_text_path = output_folder / "output_text.txt"

# YouTube URL of the video to process
# Replace this URL to process a different video
video_url = "https://youtu.be/3dhcmeOTZ_Q"

### The Four Helper Functions

The pipeline is decomposed into four single-responsibility functions. Each function does exactly one thing: download, extract frames, extract audio, or transcribe. This decomposition makes each step independently testable and easy to replace without touching the others.

### Download Video

In [ ]:
from yt_dlp import YoutubeDL

def download_video(url, output_path):
    # yt-dlp options dictionary controlling download behaviour
    # format: prefer MP4 at best quality, fall back to any format if MP4 is unavailable
    # outtmpl: the output filename template. %(ext)s is replaced with the actual extension.
    # quiet=True: suppress yt-dlp's verbose progress output
    # noplaylist=True: download only the single video, not the whole playlist if the URL is a playlist
    ydl_opts = {
        "format": "best[ext=mp4]/best",
        "outtmpl": str(output_path / "input_vid.%(ext)s"),
        "quiet": True,
        "noplaylist": True,
    }

    with YoutubeDL(ydl_opts) as ydl:
        # extract_info downloads the video and returns a metadata dictionary
        # containing title, author, view count, duration, and many other fields
        info = ydl.extract_info(url, download=True)

    # Return a clean subset of the metadata for use in the prompt later
    return {
        "title": info.get("title"),
        "author": info.get("uploader"),
        "views": info.get("view_count"),
        "duration": info.get("duration"),
    }

### Video to Images

In [ ]:
from moviepy.editor import VideoFileClip

def video_to_images(video_path, output_folder):
    # VideoFileClip reads the MP4 file and exposes it as a sequence of frames
    clip = VideoFileClip(str(video_path))

    # write_images_sequence saves frames to disk as PNG files
    # frame%04d.png produces filenames like frame0001.png, frame0002.png etc.
    # fps=0.2 means one frame is saved every 5 seconds (0.2 frames per second)
    # A 10-minute video at fps=0.2 produces 120 frames which is manageable
    # Higher fps would produce more frames and more granular visual coverage
    # but would also increase embedding time and index size proportionally
    clip.write_images_sequence(
        str(output_folder / "frame%04d.png"),
        fps=0.2
    )

### Video to Audio

In [ ]:
from moviepy.editor import VideoFileClip

def video_to_audio(video_path, output_audio_path):
    # VideoFileClip exposes the video's audio track as a separate AudioFileClip object
    clip = VideoFileClip(str(video_path))
    audio = clip.audio

    # write_audiofile saves the audio as a WAV file
    # WAV is an uncompressed format that Whisper and SpeechRecognition both handle reliably
    # MP3 would be smaller but requires additional codec support that may not be installed
    audio.write_audiofile(str(output_audio_path))

### Audio to Text

In [ ]:
from faster_whisper import WhisperModel

# Load the Whisper base model once at module level so it is not re-downloaded on every call
# device='cpu': run inference on CPU since most development machines do not have a GPU
# compute_type='int8': quantise weights to 8-bit integers which runs 2-4x faster on CPU
# with only a small reduction in transcription accuracy compared to float32
# The base model is a good balance between speed and accuracy for English speech
model = WhisperModel("base", device="cpu", compute_type="int8")

def audio_to_text(audio_path):
    # transcribe returns a generator of segment objects and an info object
    # Each segment contains a text field with the transcribed words for that time window
    segments, info = model.transcribe(str(audio_path))

    # Join all segment texts into one continuous string
    # The generator is consumed here to produce the complete transcript
    text = " ".join(segment.text for segment in segments)

    return text

### Save Text

In [ ]:
def save_text(text_data, output_text_path):
    # Path.write_text writes the string to disk in one atomic operation
    # encoding='utf-8' handles any Unicode characters in the transcript
    # including accented names, special symbols, and non-ASCII content
    output_text_path.write_text(text_data, encoding="utf-8")
    print("Text data saved successfully!")

## Category 3  Executing the Video Preprocessing Pipeline

### The Full Pipeline in Five Steps

The preprocessing pipeline transforms a YouTube URL into three artefacts that LlamaIndex can index: PNG image frames representing the visual content, a text file containing the full spoken transcript, and (temporarily) a WAV audio file that is removed once transcription is complete.

Each step depends on the output of the previous step. The video must be downloaded before frames or audio can be extracted. The audio must be extracted before it can be transcribed. The transcript must be generated before it can be saved and the audio file can be deleted.

Running the steps in strict sequence ensures that each function receives a file that already exists on disk before it tries to open it.

## Step 1  Download the YouTube Video

In [ ]:
video_url

In [ ]:
output_video_path

In [ ]:
# download_video uses yt-dlp to fetch the best available MP4 from the YouTube URL
# It saves the file to video_data/input_vid.mp4 and returns a metadata dictionary
# containing the title, author, view count, and duration of the video
# This metadata is stored and later serialised into the LLM prompt so the model
# has context about the video source when generating its answer
metadata = download_video(video_url, output_video_path)

print(metadata)

## Step 2  Extract Images from the Video

In [ ]:
# video_to_images reads the MP4 and writes one PNG frame every 5 seconds
# Each frame captures what was visible on screen at that moment in the video
# The resulting PNG files in mixed_data will later be loaded by SimpleDirectoryReader
# and embedded by CLIP into image vectors stored in the LanceDB image_collection
video_to_images(input_video_path, output_folder)

print("Frames extracted successfully!")

## Step 3  Extract Audio from the Video

In [ ]:
input_video_path

In [ ]:
output_audio_path

In [ ]:
# video_to_audio extracts the audio track from the MP4 and saves it as a WAV file
# WAV is uncompressed which avoids any lossy codec artefacts that could
# reduce Whisper transcription accuracy on quiet or fast speech segments
video_to_audio(input_video_path, output_audio_path)

print("Audio extracted successfully!")

## Step 4  Convert Audio to Text

In [ ]:
# audio_to_text calls the faster-whisper base model to transcribe the WAV file
# The model processes the audio in overlapping windows and returns timestamped segments
# All segment texts are joined into one continuous string which becomes the transcript
# This transcript captures everything the speaker said in the video and will be
# embedded as text by BAAI/bge-small-en-v1.5 and stored in the LanceDB text_collection
text_data = audio_to_text(output_audio_path)

print(text_data)

## Step 5  Save the Transcribed Text

In [ ]:
# save_text writes the transcript to mixed_data/output_text.txt
# SimpleDirectoryReader will pick up this file along with the PNG frames
# and load it as a text Document that gets embedded and stored in the text vector store
save_text(text_data, output_text_path)

## Step 6  Verify the Generated Files

In [ ]:
# List all files in mixed_data to confirm all three outputs are present:
# output_audio.wav (temporary), output_text.txt, and all frame*.png files
for file in output_folder.iterdir():
    print(file)

## Removing the Temporary Audio File

The WAV audio file served as an intermediate artefact that allowed Whisper to read the audio. Now that the transcript has been saved to output_text.txt, the WAV file is no longer needed. Removing it prevents SimpleDirectoryReader from loading the audio file into the index (audio is not a supported modality in this pipeline) and also frees disk space since WAV files for a typical YouTube video can be several hundred megabytes.

In [ ]:
# Remove the WAV audio file now that transcription is complete
# This prevents SimpleDirectoryReader from attempting to load it as a document
# and frees the disk space the uncompressed audio file occupies
os.remove(output_audio_path)
print("Audio file removed")

## Category 4  Building the Multimodal Vector Index

### Loading Mixed-Data Documents Into LlamaIndex

SimpleDirectoryReader scans the mixed_data folder and loads every file it finds. For PNG files it creates ImageDocument objects. For the text file it creates a standard Document object. Both types implement the same LlamaIndex Node interface so they can be stored and retrieved through a unified index.

### Separate LanceDB Vector Stores for Text and Images

Text and image embeddings have different dimensions and are produced by different models, so they must be stored in separate collections. A text chunk embedded by BGE produces a 384-dimensional float vector. An image frame embedded by CLIP ViT-B/32 produces a 512-dimensional float vector. LanceDB stores each collection in a separate table inside the lancedb/ folder on disk, and LlamaIndex routes each document type to the correct table automatically via the StorageContext.

### StorageContext

StorageContext.from_defaults wires together the two stores and passes them to MultiModalVectorStoreIndex. The vector_store argument receives text embeddings and the image_store argument receives image embeddings. When the index is queried it consults both stores and merges the results before returning them.

### Two Embedding Models

BAAI/bge-small-en-v1.5 embeds the transcript text. It produces 384-dimensional vectors that capture semantic meaning and are optimised for retrieval tasks.

CLIP ViT-B/32 embeds the image frames. CLIP was trained on 400 million image-caption pairs using contrastive learning so images and their natural language descriptions are mapped to nearby points in the same vector space. This means a text query like linear regression equation will retrieve frames showing a regression formula even though the frame itself contains no text that the search can match directly.

Settings.embed_model and Settings.image_embed_model are LlamaIndex global settings that tell the index which embedding model to use for each document type without having to pass the model explicitly to every function call.

## Load the Multimodal Documents

In [ ]:
from llama_index.core import SimpleDirectoryReader

# SimpleDirectoryReader scans the mixed_data folder and loads all supported file types
# PNG files become ImageDocument objects
# The text file becomes a standard Document object
# Both types implement the LlamaIndex Node interface for unified indexing
documents = SimpleDirectoryReader(output_folder).load_data()

In [ ]:
# Print the total number of documents loaded
# Should equal number of PNG frames plus 1 for the text file
len(documents)

## Import Required LlamaIndex Modules

In [ ]:
from llama_index.core.indices import MultiModalVectorStoreIndex
from llama_index.core import SimpleDirectoryReader
from llama_index.core import StorageContext

from llama_index.vector_stores.lancedb import LanceDBVectorStore

## Create LanceDB Vector Stores

In [ ]:
# text_store: LanceDB table that will store transcript chunk embeddings (384-dim from BGE)
# uri='lancedb': store all LanceDB data inside a local lancedb/ folder in the project directory
# table_name='text_collection': the specific table name inside LanceDB for text vectors
text_store = LanceDBVectorStore(uri="lancedb", table_name="text_collection")

# image_store: separate LanceDB table for image frame embeddings (512-dim from CLIP)
# Must be a separate table because image vectors have different dimensions from text vectors
image_store = LanceDBVectorStore(uri="lancedb", table_name="image_collection")

## Create a Storage Context

In [ ]:
# StorageContext wires together the text and image vector stores
# vector_store receives embeddings for text documents (from BGE)
# image_store receives embeddings for image documents (from CLIP)
# MultiModalVectorStoreIndex uses this context to route each document type
# to its correct store automatically during indexing and retrieval
storage_context = StorageContext.from_defaults(
    vector_store=text_store,
    image_store=image_store
)

## Build the MultiModalVectorStoreIndex

### 1. Text Embedding Model

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# BAAI/bge-small-en-v1.5 produces 384-dimensional vectors optimised for retrieval tasks
# Setting Settings.embed_model registers this as the global text embedding model
# LlamaIndex will call it automatically for every text Document during indexing
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

### 2. Image Embedding Model

In [ ]:
from llama_index.embeddings.clip import ClipEmbedding

# ViT-B/32 is a Vision Transformer CLIP model that produces 512-dimensional image vectors
# CLIP was trained on 400 million image-caption pairs using contrastive learning
# This means text queries and visually matching images land near each other in vector space
# A query like 'linear regression chart' can retrieve a frame showing a regression plot
# even though the frame image file itself contains no searchable text
Settings.image_embed_model = ClipEmbedding(
    model_name="ViT-B/32"
)

## Create the Multimodal Vector Index

In [ ]:
# MultiModalVectorStoreIndex.from_documents processes all loaded documents:
# For each text Document: calls Settings.embed_model to get a 384-dim BGE vector
#   and stores (text, vector) in the text_collection LanceDB table
# For each ImageDocument: calls Settings.image_embed_model to get a 512-dim CLIP vector
#   and stores (image_path, vector) in the image_collection LanceDB table
# Both stores are then ready for semantic nearest-neighbor search at query time
index = MultiModalVectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

### Index Architecture Diagram

```text
mixed_data/
│
├── output_text.txt
│   │
│   ▼
│   HuggingFaceEmbedding
│   (BAAI/bge-small-en-v1.5)
│
├── frame0001.png
├── frame0002.png
│   │
│   ▼
│   ClipEmbedding
│   (ViT-B/32)
│
▼
MultiModalVectorStoreIndex
│
▼
LanceDB
├── text_collection
└── image_collection
```

## Category 5  Multimodal Retriever, Querying, and Visualisation

### The Multimodal Retriever

index.as_retriever creates a retriever that searches both the text_collection and the image_collection simultaneously when a query arrives.

similarity_top_k=3 means the retriever returns the 3 most similar text chunks. The query text is embedded by BGE into a 384-dimensional vector and the closest 3 text vectors in the text_collection are selected using cosine similarity.

image_similarity_top_k=5 means the retriever also returns the 5 most visually similar image frames. The query text is embedded by CLIP into a 512-dimensional vector and the closest 5 image vectors in the image_collection are selected. CLIP's joint text-image embedding space means a text query finds relevant frames even without any visual feature matching.

### Inspecting Retrieved Nodes

Each retrieved object is a NodeWithScore containing a Node and a similarity score. The node.metadata dictionary holds the file_path key which identifies which PNG frame or text file the node came from. Printing scores and metadata lets us verify the retriever is returning genuinely relevant content before passing it to the LLM.

### Separating Text and Image Results

The retrieve helper function iterates over all retrieved nodes and checks the type of each node. Nodes that are ImageNode instances have their file_path added to the retrieved_images list. All other nodes are text nodes and their text content is added to retrieved_text. This separation is necessary because the LLM prompt needs the text as a context string and the images as separate PIL Image objects.

## Create the Multimodal Retriever

In [ ]:
# as_retriever wraps the MultiModalVectorStoreIndex as a retriever object
# similarity_top_k=3: return the 3 most semantically similar text chunks per query
# image_similarity_top_k=5: return the 5 most visually relevant image frames per query
# Both searches run against their respective LanceDB collections simultaneously
retriever = index.as_retriever(
    similarity_top_k=3,
    image_similarity_top_k=5
)

## Retrieve Relevant Results

In [ ]:
# Test the retriever with a general question about the video content
# The retriever embeds this query with both BGE (for text search) and CLIP (for image search)
# and returns the most relevant nodes from each modality
query = "Explain the concept discussed in the video."

retrieved_nodes = retriever.retrieve(query)

In [ ]:
# Inspect the raw retrieved nodes before displaying them
retrieved_nodes

## Display the Retrieved Results

In [ ]:
# Print each retrieved node with its type, content summary, and metadata
# The separator line makes it easier to distinguish where one node ends and the next begins
for node in retrieved_nodes:
    print(node)
    print("-" * 80)

### Querying the Retriever for Visual Content

In [ ]:
# This query targets the visual content specifically
# CLIP embeds 'objects, charts, or diagrams' as a text vector
# and retrieves the frames whose CLIP image vectors are closest to it
# This demonstrates that text queries can retrieve visually relevant frames
# even when the frame images themselves contain no searchable text metadata
query = "What objects, charts, or diagrams are shown in the video?"

retrieved_nodes = retriever.retrieve(query)

In [ ]:
# Print all retrieved nodes for the visual content query
for node in retrieved_nodes:
    print(node)
    print("-" * 80)

In [ ]:
# Print the file_path metadata of each retrieved node
# This shows which specific PNG frame files were selected by the retriever
for node in retrieved_nodes:
    print(node.metadata.get("file_path"))

In [ ]:
# Print the similarity score for each retrieved node
# Higher scores indicate the node's embedding is closer to the query embedding
# Scores can be used to set a threshold below which results are considered irrelevant
for node in retrieved_nodes:
    print(f"Similarity Score: {node.score:.4f}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Display the retrieved image frames inline so we can visually verify
# that the retriever selected frames showing relevant visual content
for node in retrieved_nodes:
    file_path = node.metadata.get("file_path")

    # Only display nodes that point to image files (PNG or JPEG)
    if file_path and file_path.endswith((".png", ".jpg", ".jpeg")):
        img = Image.open(file_path)

        plt.imshow(img)
        plt.axis("off")
        plt.title(file_path)
        plt.show()

## Create a Retrieval Helper Function

In [ ]:
from llama_index.core.response.notebook_utils import display_source_node
from llama_index.core.schema import ImageNode

In [ ]:
def retrieve(retriever, query):
    # Run the retriever against both the text and image vector stores
    retrieval_results = retriever.retrieve(query)

    retrieved_images = []
    retrieved_text = []

    for result in retrieval_results:
        # isinstance check separates image nodes from text nodes
        # ImageNode objects have an image_path or file_path in their metadata
        # Text nodes have a text attribute containing the chunk content
        if isinstance(result.node, ImageNode):
            # Collect the file path so the image can be opened with PIL later
            retrieved_images.append(result.node.metadata["file_path"])
        else:
            # Collect the text content to build the context string for the LLM
            retrieved_text.append(result.node.get_content())

    return retrieved_images, retrieved_text

## Retrieve Relevant Text and Images

In [ ]:
# Ask a specific question about a concept that should appear in both the transcript
# (as spoken explanation) and in the image frames (as slides or diagrams)
query = "Can you explain linear regression and the equation of multiple linear regression?"

# retrieve returns two lists: file paths of relevant frames and text chunks from the transcript
images, text = retrieve(retriever, query)

## Visualize the Retrieved Images

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

def plot_images(image_paths, max_images=5):
    # Create a wide figure to display multiple images side by side
    plt.figure(figsize=(16, 9))

    images_shown = 0

    for image_path in image_paths:
        # os.path.isfile verifies the file exists before attempting to open it
        # A missing file would raise a FileNotFoundError without this check
        if os.path.isfile(image_path):

            image = Image.open(image_path)

            # Create a subplot position for each image in a single row
            plt.subplot(1, min(len(image_paths), max_images), images_shown + 1)
            plt.imshow(image)
            plt.axis("off")

            images_shown += 1

            if images_shown >= max_images:
                break

    plt.tight_layout()
    plt.show()

## Display the Retrieved Images

In [ ]:
# Visualise the top retrieved frames before passing them to the LLM
# This lets us verify the retriever found frames showing regression content
# rather than unrelated visual content from other parts of the video
plot_images(images)

## Category 6  Multimodal Prompt, Gemini Integration, and Answer Generation

### Why the Google GenAI SDK Instead of GeminiMultiModal From LlamaIndex

The notebook originally intended to use LlamaIndex's GeminiMultiModal wrapper which provides a clean integrate-once interface. However with current package versions (llama-index-core 0.14.x), the wrapper has a compatibility issue where it does not correctly handle the ImageDocument objects that LlamaIndex creates from PNG files, resulting in errors during the generate call.

The official Google GenAI SDK (google-genai) has no such issue. It accepts a list of mixed content items where each item can be either a string or a PIL Image object. We build this list manually by putting the text prompt first and then appending each retrieved PIL Image. The SDK sends everything to Gemini 2.5 Flash which natively understands both text and images in the same input.

### The Prompt Template

The prompt explicitly instructs the model to use only the retrieved context and images. This is the standard instruction for grounded RAG generation: constraining the model to the provided context reduces hallucination because the model cannot fall back on its training knowledge when the retrieved context is insufficient. The prompt includes three placeholders: context_str for the joined text chunks, metadata_str for the video metadata JSON, and query_str for the user question.

### Building the Contents List for Gemini

client.models.generate_content accepts a contents list where each element is either a string or an image object. The first element is the complete text prompt. Then for each retrieved node whose file_path ends in .png, a PIL Image opened from that path is appended to the list. Gemini reads all elements in order, combining the instructions and context from the text prompt with the visual evidence from the images to generate a grounded multimodal answer.

## Create the Multimodal Prompt Template

In [ ]:
# The prompt template has three placeholders:
# {context_str}: the retrieved text chunks joined into one string
# {metadata_str}: the video metadata serialised as JSON (title, author, duration)
# {query_str}: the user's question
# The explicit 'ONLY' instruction prevents the model from using its training knowledge
# beyond the retrieved content, which is the standard grounding technique in RAG
qa_prompt = """
You are a helpful AI assistant.

Use only the provided retrieved text, images, and video metadata to answer the user's question.

Instructions:
Use the retrieved text and images together to answer the question.
Do not use outside knowledge or make assumptions.
If the answer cannot be found in the provided information, say that you do not know.

Retrieved Context:
{context_str}

Video Metadata:
{metadata_str}

User Question:
{query_str}

Answer:
"""

## Convert the Video Metadata to JSON

In [ ]:
import json

# Serialise the metadata dictionary into a formatted JSON string
# indent=2 produces human-readable indentation in the JSON output
# The string is included in the prompt so Gemini knows the video title, author,
# and duration when generating its answer, providing useful grounding context
metadata_str = json.dumps(metadata, indent=2)

In [ ]:
# Verify that metadata_str is a plain Python string
# The prompt format call requires a string, not a dict or any other type
print(type(metadata_str))

## Prepare the Context String

In [ ]:
# Join all retrieved text chunks into one continuous context string
# Double newlines between chunks help Gemini identify paragraph boundaries
# This string fills the {context_str} placeholder in the prompt template
context_str = "\n\n".join(text)

In [ ]:
from IPython.display import Markdown, display

# Render the context string as Markdown so any formatting in the transcript is visible
display(Markdown(context_str))

## Load Retrieved Images

In [ ]:
# Verify the type of the last PIL Image opened
# Should be PIL.PngImagePlugin.PngImageFile or similar
print(type(img))

In [ ]:
# Build a list of file paths for all PNG files among the last retrieved nodes
# These are the frames that will be sent alongside the text prompt to Gemini
image_paths = [
    node.metadata["file_path"]
    for node in retrieved_nodes
    if node.metadata["file_path"].endswith(".png")
]

print(image_paths)

In [ ]:
from llama_index.core import SimpleDirectoryReader

# Load the selected image files as LlamaIndex ImageDocuments
# input_files accepts an explicit list of paths rather than scanning a directory
# This was used for the LlamaIndex GeminiMultiModal approach which has compatibility issues
# The Google GenAI SDK approach below uses PIL Image objects directly instead
image_documents = SimpleDirectoryReader(
    input_files=image_paths
).load_data()

## Initialize the Gemini Multimodal LLM

In [ ]:
import os
from dotenv import load_dotenv

# Load the Google API key from the .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [ ]:
from llama_index.multi_modal_llms.gemini import GeminiMultiModal

# GeminiMultiModal is the LlamaIndex wrapper for the Gemini vision models
# model_name: gemini-2.5-flash is the fast efficient multimodal model
# NOTE: this wrapper has compatibility issues with current LlamaIndex ImageDocument objects
# The Google GenAI SDK call below is the working alternative used in this notebook
gemini_mm_llm = GeminiMultiModal(
    model_name="models/gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
)

## Define the User Query

In [ ]:
# The specific question that will be answered using retrieved text and image frames
query_str = (
    "Can you explain what Linear Regression is and "
    "what the equation of Linear Regression is?"
)

## Generate the Response Using Gemini (LlamaIndex wrapper attempt)

In [ ]:
# This call uses the LlamaIndex GeminiMultiModal wrapper
# It may raise a compatibility error with current package versions
# The working alternative using the Google GenAI SDK is shown in the cells below
response = gemini_mm_llm.complete(
    prompt=qa_prompt.format(
        context_str=context_str,
        metadata_str=metadata_str,
        query_str=query_str,
    ),
    image_documents=image_documents,
)

print(response.text)

## Perform Multimodal Inference With Gemini Using the Official Google GenAI SDK

The GeminiMultiModal wrapper in LlamaIndex has compatibility issues with current ImageDocument objects. The official Google GenAI SDK provides a direct and reliable alternative that accepts a mixed list of text strings and PIL Image objects in a single contents list. This approach gives full control over exactly which images are sent to the model and in what order.

## Import Required Libraries

In [ ]:
from google import genai
from google.genai import types
from PIL import Image

## Initialize the Gemini Client

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# genai.Client is the official Google GenAI SDK client
# It authenticates using the API key and routes all requests to the Gemini API
client = genai.Client(api_key=GOOGLE_API_KEY)

## Create the Final Prompt

In [ ]:
# This is the formatted prompt string that will be the first element in the contents list
# It combines the retrieved text context, the video metadata JSON, and the user question
# into one structured instruction block for Gemini to read
qa_prompt = f"""
You are a helpful AI assistant.

Use ONLY the retrieved context, retrieved images and video metadata to answer the user's question.

Instructions:
Use both the retrieved text and retrieved images.
Do not use external knowledge.
If the answer is not present in the provided information, say that you do not know.

Retrieved Context:
{context_str}

Video Metadata:
{metadata_str}

User Question:
{query_str}

Answer:
"""

## Prepare the Retrieved Images for Gemini

In [ ]:
from PIL import Image

# Build the contents list that will be sent to Gemini
# The list can contain a mix of strings and PIL Image objects
# Gemini reads them in order so the text prompt with all instructions comes first
contents = [qa_prompt]

# Iterate through all retrieved nodes and append any image frames as PIL Image objects
for node in retrieved_nodes:
    # metadata['file_path'] holds the path to the frame PNG on disk
    file_path = node.metadata.get("file_path", "")

    # Only append nodes that point to actual image files
    if file_path.endswith((".png", ".jpg", ".jpeg")):
        # PIL.Image.open loads the pixel data from disk
        # Gemini accepts PIL Image objects directly in the contents list
        img = Image.open(file_path)
        contents.append(img)

## Generate the Final Answer

In [ ]:
# Send the full contents list to Gemini 2.5 Flash
# Gemini reads the text prompt, all retrieved images, and generates a grounded answer
# The model can reference specific details visible in the frames because it sees them
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=contents,
)

## Display the Response

In [ ]:
# Print the final generated answer from Gemini
# The answer should reference both the spoken explanation from the transcript
# and the visual content shown in the retrieved frames
print(response.text)

## Category 7  Pipeline Architecture and Conclusion

### Full Preprocessing Pipeline Architecture

```text
                 Video
                   │
        ┌──────────┴──────────┐
        │                     │
     Extract Frames      Extract Audio
        │                     │
        │              Speech-to-Text
        └──────────┬──────────┘
                   │
        Text + Image Documents
                   │
     MultiModalVectorStoreIndex
        ┌──────────┴──────────┐
        │                     │
  text_collection       image_collection
   (BGE 384-dim)        (CLIP 512-dim)
```

### Query Execution Flow

Once the Multimodal Vector Index has been built, every user question triggers this workflow.

The retriever embeds the question using both BGE (for text search) and CLIP (for image search) simultaneously. It searches the text_collection for the 3 most semantically similar transcript chunks and searches the image_collection for the 5 most visually relevant frames.

The retrieved text chunks are joined into context_str. The retrieved frames are opened as PIL Image objects. Both are combined with the structured prompt and the video metadata into the contents list.

Gemini 2.5 Flash receives the full contents list and generates a grounded answer that draws on both the spoken content and the visual evidence shown in the retrieved frames.